# 🚀 FEDHEP POST-MIDTERM EXPERIMENTS — PART 2 OF 2
## 🛡️ STUDY B & C: Advanced Byzantine Attacks & Majority Breakdown ($q \ge 50\%$)

> **Mục tiêu nghiên cứu**:
> 1. **Study C (Idea C)**: Kiểm tra khả năng phòng thủ của FedHEP trước các kỹ thuật tấn công tinh vi nhắm vào Temporal Trust Tracking (TTT) và Subspace Cosine Filtering (*IPM*, *Adaptive Cosine*, *Delayed Onset*).
> 2. **Study B (Idea B)**: Khảo sát hiện tượng lật ngược đồng thuận (Consensus Inversion) khi kẻ tấn công chiếm đa số ($q \in [0.1, 0.7]$) so sánh trực tiếp với **FedAvg** và **Ditto**.
>
> ⚡ **Chạy song song**: Notebook này chạy độc lập với **Part 1** trên Kaggle GPU để tối ưu thời gian chạy.

In [ ]:
import os
import sys

# 1. Repository Setup
ROOT = '/kaggle/working/Topology-aware-FDL'
if not os.path.exists(ROOT):
    print('🚀 Cloning repository from GitHub...')
    !git clone https://github.com/nam200718/Topology-aware-FDL.git {ROOT}
else:
    print('🔄 Repository exists, pulling latest updates from main...')
    !cd {ROOT} && git pull origin main

%cd {ROOT}
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 2. DEDICATED OUTPUT DIRECTORY (Outside of repo, directly visible in Kaggle Output)
if os.path.exists('/kaggle/working'):
    OUT_DIR = '/kaggle/working/oreh_results_part2'
else:
    OUT_DIR = os.path.abspath('./oreh_results_part2')

SANITY_DIR = os.path.join(OUT_DIR, 'sanity')
EVAL_DIR = os.path.join(OUT_DIR, 'eval')
FIG_DIR = os.path.join(OUT_DIR, 'figures')

os.makedirs(SANITY_DIR, exist_ok=True)
os.makedirs(EVAL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('✅ Working Directory:', os.getcwd())
print('📁 Dedicated Results Directory:', OUT_DIR)
print('📊 CSV Tables Directory:', EVAL_DIR)
print('📈 Figures Directory:', FIG_DIR)

# 3. ROBUST OUTPUT & EMERGENCY PACKAGING (Guarantees output saved even if timeout / limit reached)
import shutil
import atexit
import signal

ZIP_BASE = '/kaggle/working/oreh_part2_attacks_summary' if os.path.exists('/kaggle/working') else './oreh_part2_attacks_summary'

def package_results_zip():
    try:
        if os.path.exists(OUT_DIR) and os.listdir(OUT_DIR):
            shutil.make_archive(ZIP_BASE, 'zip', OUT_DIR)
            print(f'📦 [Auto-Package] Output archive updated: {ZIP_BASE}.zip')
    except Exception as e:
        print(f'⚠️ Warning during packaging: {e}')

# Register exit handlers for process termination or Kaggle 12h timeout / cancellation signals
atexit.register(package_results_zip)
try:
    signal.signal(signal.SIGTERM, lambda s, f: (package_results_zip(), sys.exit(0)))
    signal.signal(signal.SIGINT, lambda s, f: (package_results_zip(), sys.exit(0)))
except Exception:
    pass

In [ ]:
!pip -q install pydantic pyyaml torchvision pandas matplotlib seaborn scipy
import torch

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'VRAM Capacity: {vram_gb:.2f} GB')
    torch.backends.cudnn.benchmark = True
else:
    print('⚠️ No GPU detected! Please navigate to Notebook Settings -> Accelerator -> GPU T4 x1.')

In [ ]:
from src.data.dataset import _auto_link_dataset

data_dir = './data'
found = _auto_link_dataset(data_dir, 'cifar100')
if found:
    print('⚡ [Kaggle Input] CIFAR-100 dataset successfully verified and linked! (0.0s latency, 0 downloads)')
else:
    print('ℹ️ Note: If CIFAR-100 is attached under /kaggle/input, the experiment runner will automatically link it.')

In [ ]:
# ==============================================================================
# Cell 4: 1-Round Pre-Flight Sanity Check
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running 1-round pre-flight sanity check on CIFAR-100 using {device}...')

sanity_runner = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=1,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=1000,
    test_subset=500,
    data_dir=data_dir,
    output_dir=SANITY_DIR,
    clustering_mode='none',
)

for test_atk in ['ipm', 'adaptive_cosine', 'delayed_onset']:
    test_res = sanity_runner.run_single_experiment(
        defense_scope='both',
        attack_target_tier='all',
        attack_type=test_atk,
        byzantine_rate=0.2,
        seed=42,
    )
    print(f'✅ Sanity Check {test_atk.upper()} PASSED! Acc: {test_res["final_test_acc"]}% | Byz Trust: {test_res["byzantine_trust_score"]} | Latency: {test_res["total_defense_latency_ms"]} ms')

print('\n🚀 All Pre-Flight Sanity Checks Successfully Completed! Ready for Study C & B.')

## 🧪 STUDY C (Idea C): New Attacks Suite Targeting Temporal Trust Tracking
Đánh giá khả năng phòng thủ trước các cuộc tấn công nâng cao:
* **IPM (Inner Product Manipulation)**: duy trì cosine nhỏ dương với honest delta ($\epsilon = 0.1$).
* **Adaptive Cosine**: chủ động lật dấu luân phiên để tránh bị triệt tiêu uy tín.
* **Delayed Onset**: tấn công sau round 15 (giai đoạn đầu hoàn toàn trung thực để tích lũy Trust Score cao).
* **Baselines**: `sign_flip`, `alie`.
* **Output File**: `/kaggle/working/oreh_results_part2/eval/cifar100_postmid_attacks.csv`

In [ ]:
# ==============================================================================
# Cell 6: [STUDY C] New Attacks Suite Targeting TTT
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_c = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
    clustering_mode='none',
)

df_attacks = runner_c.run_postmid_attacks(
    attacks=['sign_flip', 'alie', 'ipm', 'adaptive_cosine', 'delayed_onset'],
    byzantine_rates=[0.2, 0.3],
    seeds=[42, 123, 7],
    defense_scope='both',
    attack_target_tier='all',
    ipm_epsilon=0.1,
    attack_onset_round=15,
)

print('\n--- STUDY C (NEW ATTACKS SUITE) SUMMARY TABLE ---')
cols_c = ['method', 'attack_type', 'byzantine_rate', 'seed', 'final_test_acc', 'bottom_10_fairness', 'byzantine_trust_score', 'honest_trust_score']
if not df_attacks.empty:
    print(df_attacks[[c for c in cols_c if c in df_attacks.columns]].to_string(index=False))

# Save progress and update zip archive immediately
package_results_zip()

## 🛡️ STUDY B (Idea B): Majority Attacker Stress-Test ($q \ge 50\%$)
Khảo sát xem khi tỷ lệ tấn công vượt ngưỡng đa số ($q \in \{0.5, 0.6, 0.7\}$), tâm cụm của Subspace Cosine Filtering có bị lật ngược (Consensus Inversion) không, và FedHEP so với FedAvg / Ditto ra sao.
* **Methods**: **Defended FedHEP**, **FedAvg**, **Ditto**
* **Attacker Fractions**: $q \in \{0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7\}$
* **Output File**: `/kaggle/working/oreh_results_part2/eval/cifar100_postmid_majority_q.csv`

In [ ]:
# ==============================================================================
# Cell 8: [STUDY B] Majority Attacker Stress-Test (q >= 50%)
# ==============================================================================
from oreh.comparison_runner import OrehComparisonRunner
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
runner_b = OrehComparisonRunner(
    dataset='cifar100',
    num_rounds=30,
    num_clients=15,
    num_clusters=3,
    alpha=0.5,
    device=device,
    train_subset=10000,
    test_subset=3000,
    data_dir=data_dir,
    output_dir=EVAL_DIR,
    clustering_mode='none',
)

df_majority = runner_b.run_postmid_majority_q(
    byzantine_rates=[0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7],
    attacks=['sign_flip', 'alie'],
    methods=['fedhep', 'fedavg', 'ditto'],
    seeds=[42, 123],
    defense_scope='both',
)

print('\n--- STUDY B (MAJORITY q >= 50%) SUMMARY TABLE ---')
cols_b = ['method', 'attack_type', 'byzantine_rate', 'seed', 'final_test_acc', 'bottom_10_fairness', 'byzantine_trust_score']
if not df_majority.empty:
    print(df_majority[[c for c in cols_b if c in df_majority.columns]].to_string(index=False))

# Save progress and update zip archive immediately
package_results_zip()

## 📊 Publication-Quality Visualizations & Statistical Analysis
Vẽ biểu đồ so sánh các loại attack mới (Study C) và đường cong suy giảm khi $q \ge 50\%$ (Study B).

In [ ]:
# ==============================================================================
# Cell 10: Plotting Study C & Study B Results
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Plot Study C: Attacks Comparison
attacks_file = os.path.join(EVAL_DIR, 'cifar100_postmid_attacks.csv')
if os.path.exists(attacks_file):
    df_c = pd.read_csv(attacks_file)
    if not df_c.empty:
        plt.figure(figsize=(10, 5.5))
        summary_c = df_c.groupby(['attack_type', 'byzantine_rate'])['final_test_acc'].agg(['mean', 'std']).reset_index()
        atks = summary_c['attack_type'].unique()
        q_vals = summary_c['byzantine_rate'].unique()
        x = np.arange(len(atks))
        width = 0.35
        for i, q in enumerate(sorted(q_vals)):
            sub = summary_c[summary_c['byzantine_rate'] == q]
            acc_map = dict(zip(sub['attack_type'], sub['mean']))
            std_map = dict(zip(sub['attack_type'], sub['std'].fillna(0)))
            means = [acc_map.get(a, 0.0) for a in atks]
            stds = [std_map.get(a, 0.0) for a in atks]
            plt.bar(x + i * width, means, width, yerr=stds, capsize=4, label=f'q = {q}', alpha=0.85)
        plt.title('CIFAR-100: Robustness under Advanced Byzantine Attacks Targeting TTT', fontsize=12, fontweight='bold')
        plt.xlabel('Attack Type', fontsize=11)
        plt.ylabel('Personalized Test Accuracy (%)', fontsize=11)
        plt.xticks(x + width / 2, [a.replace('_', ' ').title() for a in atks], fontsize=10)
        plt.grid(axis='y', linestyle='--', alpha=0.6)
        plt.legend(frameon=True)
        plt.tight_layout()
        fig_c = os.path.join(FIG_DIR, 'postmid_c_attacks_comparison.png')
        plt.savefig(fig_c, dpi=200)
        plt.show()
        print(f'✅ Saved Study C plot to: {fig_c}')

# 2. Plot Study B: Majority Attacker Inversion Curve (Sign-Flip & ALIE)
majority_file = os.path.join(EVAL_DIR, 'cifar100_postmid_majority_q.csv')
if os.path.exists(majority_file):
    df_b = pd.read_csv(majority_file)
    if not df_b.empty:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
        colors = {'defended fedhep': '#1f77b4', 'fedhep': '#1f77b4', 'fedavg': '#d62728', 'ditto': '#2ca02c'}
        for ax, atk_name in zip(axes, ['sign_flip', 'alie']):
            df_sub = df_b[df_b['attack_type'].isin([atk_name, 'none'])]
            summary_b = df_sub.groupby(['method', 'byzantine_rate'])['final_test_acc'].mean().reset_index()
            for m in summary_b['method'].unique():
                sub = summary_b[summary_b['method'] == m].sort_values('byzantine_rate')
                ax.plot(sub['byzantine_rate'], sub['final_test_acc'], marker='o', linewidth=2.5,
                        label=m, color=colors.get(str(m).lower()))
            ax.axvline(x=0.5, color='crimson', linestyle=':', linewidth=2, label='q = 50% Majority')
            ax.set_title(f'CIFAR-100: Majority Breakdown ({atk_name.replace("_", " ").upper()})', fontsize=11, fontweight='bold')
            ax.set_xlabel('Attacker Fraction (q)', fontsize=10)
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.legend(frameon=True, fontsize=9)
        axes[0].set_ylabel('Personalized Test Accuracy (%)', fontsize=11)
        plt.tight_layout()
        fig_b = os.path.join(FIG_DIR, 'postmid_b_majority_q_curve.png')
        plt.savefig(fig_b, dpi=200)
        plt.show()
        print(f'✅ Saved Study B plot to: {fig_b}')

## 📦 1-Click ZIP Packaging for Direct Download
Tự động nén toàn bộ kết quả vào file `/kaggle/working/oreh_part2_attacks_summary.zip` để tải về chỉ với 1 click.

In [ ]:
# ==============================================================================
# Cell 12: Final Packaging for Direct Download
# ==============================================================================
package_results_zip()

print('\n🎉 PART 2 ALL EXPERIMENTS & PACKAGING COMPLETED!')
print(f'📁 Dedicated Folder: {OUT_DIR}')
print(f'📦 Direct Download File: {ZIP_BASE}.zip')
print('You can download oreh_part2_attacks_summary.zip directly from the Kaggle Output sidebar!')